<a href="https://colab.research.google.com/github/kotwalurja-03/Google-Collab-Project/blob/main/hospital_charges_collab_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

!pip install statsmodels tensorflow scikit-learn plotly -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

print("All libraries ready!")


In [ ]:
import re
import sqlite3

import pandas as pd
from google.colab import files

In [ ]:
uploaded = files.upload()
excel_file = next(iter(uploaded))


Saving hospital-charges.xlsx to hospital-charges.xlsx


In [ ]:
# Load the only sheet in the workbook.
df = pd.read_excel(excel_file, sheet_name="hospital-charges")


In [ ]:
def clean_column_name(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


df.columns = [clean_column_name(col) for col in df.columns]

In [ ]:
# Convert currency columns from strings such as "$32963.07" to numeric values.

# Re-apply column name cleaning to ensure consistency
df.columns = [clean_column_name(col) for col in df.columns]

money_columns = [
    "average_covered_charges",
    "average_total_payments",
    "average_medicare_payments",
]

for col in money_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

df["total_discharges"] = pd.to_numeric(df["total_discharges"], errors="coerce")
df["provider_id"] = pd.to_numeric(df["provider_id"], errors="coerce")
df["provider_zip_code"] = pd.to_numeric(df["provider_zip_code"], errors="coerce")

In [ ]:
# Create an in-memory SQLite database and save the spreadsheet as a SQL table.
conn = sqlite3.connect(":memory:")
df.to_sql("hospital_charges", conn, index=False, if_exists="replace")


def run_sql(query):
    return pd.read_sql_query(query, conn)

In [ ]:
# Preview the table.
run_sql("SELECT * FROM hospital_charges LIMIT 5")

,drg_definition,provider_id,provider_name,provider_street_address,provider_city,provider_state,provider_zip_code,hospital_referral_region_description,total_discharges,average_covered_charges,average_total_payments,average_medicare_payments
0,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,10001,SOUTHEAST ALABAMA MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,AL - Dothan,91,32963.07,5777.24,4763.73
1,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,10005,MARSHALL MEDICAL CENTER SOUTH,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,AL - Birmingham,14,15131.85,5787.57,4976.71
2,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,10006,ELIZA COFFEE MEMORIAL HOSPITAL,205 MARENGO STREET,FLORENCE,AL,35631,AL - Birmingham,24,37560.37,5434.95,4453.79
3,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,10011,ST VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,AL - Birmingham,25,13998.28,5417.56,4129.16
4,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,10016,SHELBY BAPTIST MEDICAL CENTER,1000 FIRST STREET NORTH,ALABASTER,AL,35007,AL - Birmingham,18,31633.27,5658.33,4851.44


In [ ]:

# Run SQL queries

# 1. Total rows in the dataset
query_1 = """
SELECT COUNT(*) AS total_rows
FROM hospital_charges;
"""
display(run_sql(query_1))

,total_rows
0,163065


In [ ]:

# 2. Average covered charges, total payments, and Medicare payments by state
query_2 = """
SELECT
    provider_state,
    ROUND(AVG(average_covered_charges), 2) AS avg_covered_charges,
    ROUND(AVG(average_total_payments), 2) AS avg_total_payments,
    ROUND(AVG(average_medicare_payments), 2) AS avg_medicare_payments
FROM hospital_charges
GROUP BY provider_state
ORDER BY avg_covered_charges DESC;
"""
display(run_sql(query_2))

,provider_state,avg_covered_charges,avg_total_payments,avg_medicare_payments
0,CA,67508.62,12629.67,11494.38
1,NJ,66125.69,10678.99,9586.94
2,NV,61047.12,10291.72,8747.60
3,FL,46016.23,8826.99,7667.48
4,TX,41480.19,9243.98,7970.43
5,AZ,41200.06,10154.53,8825.72
6,CO,41095.14,9502.69,8150.93
7,AK,40348.74,14572.39,12958.97
8,DC,40116.66,12998.03,11811.97
9,PA,39633.96,9100.04,7919.18


In [ ]:
# 3. Top 10 hospitals with the highest average covered charges
query_3 = """
SELECT
    provider_name,
    provider_city,
    provider_state,
    ROUND(AVG(average_covered_charges), 2) AS avg_covered_charges
FROM hospital_charges
GROUP BY provider_name, provider_city, provider_state
ORDER BY avg_covered_charges DESC
LIMIT 10;
"""
display(run_sql(query_3))


,provider_name,provider_city,provider_state,avg_covered_charges
0,UVA HEALTH SCIENCES CENTER,CHARLOTTESVILLE,VA,211922.00
1,BAYONNE HOSPITAL CENTER,BAYONNE,NJ,147441.33
2,DOCTORS MEDICAL CENTER,MODESTO,CA,144695.83
3,STANFORD HOSPITAL,STANFORD,CA,138818.65
4,NORTHBAY MEDICAL CENTER,FAIRFIELD,CA,138504.55
5,CROZER CHESTER MEDICAL CENTER,UPLAND,PA,137130.85
6,WASHINGTON HOSPITAL,FREMONT,CA,131510.12
7,SETON MEDICAL CENTER,DALY CITY,CA,130177.36
8,TEMPLE UNIVERSITY HOSPITAL,PHILADELPHIA,PA,126824.33
9,REGIONAL MEDICAL CENTER OF SAN JOSE,SAN JOSE,CA,126288.69


In [ ]:
# 4. Top 10 DRG procedures by average total payment
query_4 = """
SELECT
    drg_definition,
    ROUND(AVG(average_total_payments), 2) AS avg_total_payments,
    ROUND(AVG(average_medicare_payments), 2) AS avg_medicare_payments
FROM hospital_charges
GROUP BY drg_definition
ORDER BY avg_total_payments DESC
LIMIT 10;
"""
display(run_sql(query_4))


NameError: name 'run_sql' is not defined